# 🧮 PyTorch: derivatives as functions

*Section 15.1.2 of the notes.*

PyTorch offers its derivatives through two interfaces, and they differ in
**where the result goes**.

* `backward()` (previous notebook) returns nothing. It **writes** the
  gradients into the `.grad` of the tensors, where they stay. Code like this
  is **stateful**: calling it changes something that outlives the call, so the
  same call made twice need not give the same answer.
* the functions of the module `torch.func` take a function $f$ and **return a
  new function**, such as $f'$. Nothing is written anywhere. Code like this is **stateless**:
  everything it produces comes back as its result.

**What `torch.func` is.** A module of PyTorch whose functions take a Python
function and return a new Python function. The input is an ordinary function
written with `torch` operations, such as `def f(x): return torch.sin(x**2)`.
The ones used here:

| call | returns the function | condition |
|---|---|---|
| `grad(f)` | $x \mapsto \nabla f(x)$, by one reverse sweep | `f` returns one number |
| `vmap(f)` | $f$ applied to every entry of a batch | |
| `jacrev(f)`, `jacfwd(f)` | $x \mapsto$ the Jacobian of $f$ at $x$, by reverse or forward sweeps | |
| `hessian(f)` | $x \mapsto$ the matrix of second derivatives of $f$ at $x$ | `f` returns one number |

The result is again an ordinary function, so these calls can be combined:
`grad(grad(f))` is the second derivative, and `vmap(grad(f))` is the
derivative at every point of a batch.

This notebook shows the stateless interface at work: derivatives of
derivatives, batches without loops, full Jacobians, and the second derivatives
of a network.

## ⚙️ Setup

Timings use `%timeit -o -q` and `.best`, as in notebook `04-measuring` of
Chapter 13: the smallest time over several runs.

`torch.func` uses an internal PyTorch feature that prints a warning on every
call. The `warnings.filterwarnings` line hides that warning.

In [ ]:
import warnings

import torch
import torch.nn as nn
from torch.func import grad, hessian, jacfwd, jacrev, vmap

torch.set_default_dtype(torch.float64)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")  # internal to torch

## 1. 🔁 The same call, the same answer

Take $f(x) = \sin(x^2)$, with $f'(1) = 2\cos 1 \approx 1.0806$. We ask for
$f'(1)$ twice, once with each interface.

🤔 *Guess: which interface gives a different number the second time?*

In [ ]:
def f(x):
    return torch.sin(x**2)


x = torch.tensor(1.0, requires_grad=True)
for call in (1, 2):
    f(x).backward()
    print(f"backward, call {call}:  x.grad = {x.grad.item():.4f}")

df = grad(f)  # a new function: x -> f'(x)
one = torch.tensor(1.0)
for call in (1, 2):
    value = df(one)
    print(f"grad(f),  call {call}:  df(1)  = {value.item():.4f}")

💡 **What happened?** `backward` added its second result to the first, so the
stored value doubled. `grad(f)` is an ordinary function: the same argument
always gives the same result.

`grad(f)` is one reverse sweep (Section 3.3), packaged as a function. Since
it is a function like any other, it can be transformed again.

In [ ]:
d2f = grad(grad(f))  # the second derivative, as a function
d2f_one = d2f(one)
formula = 2 * torch.cos(one) - 4 * torch.sin(one)  # f''(1) = 2 cos 1 - 4 sin 1

print("f''(1) with grad(grad(f)):", d2f_one.item())
print("f''(1) by the formula:    ", formula.item())

## 2. 📦 A batch without a loop

`grad(f)` needs `f` to return **one number**. Its input can be an array, and
then `grad(f)` returns the gradient, an array of the same shape. Our $f$ acts
entry by entry: at 10 000 points it returns 10 000 numbers.

🤔 *Guess: what does `grad(f)` do with an array of 10 000 points?*

In [ ]:
xs = torch.linspace(0, 2, 10_000)
print("f(xs) has shape", tuple(f(xs).shape))

try:
    df(xs)
except RuntimeError as error:
    print("RuntimeError:", error)

💡 **What happened?** $f$ returned 10 000 numbers, and `grad` accepts only a
function that returns one number. The derivative of a function with many
outputs is a Jacobian, the subject of Section 3.

To get $f'$ at many points we could loop over them in Python. `vmap` does it
differently: it turns a function of one point into a function of a whole batch
of points, and runs it as array operations.

* The loop calls `df` once per point. `torch.stack` turns the list of 10 000
  results into one array.
* `vmap(df)(xs)` gives the same array in one call.
* `torch.allclose(a, b)` checks that two arrays agree up to rounding.

🤔 *Guess: for 10 000 points, how much faster is `vmap` than the Python loop?*

In [ ]:
loop = torch.stack([df(xi) for xi in xs])
batch = vmap(df)(xs)
formula = 2 * xs * torch.cos(xs**2)  # f'(x) = 2x cos(x^2)

print("loop and vmap agree:   ", torch.allclose(loop, batch))
print("vmap and formula agree:", torch.allclose(batch, formula))

t_loop = %timeit -o -q torch.stack([df(xi) for xi in xs])
t_vmap = %timeit -o -q vmap(df)(xs)
ratio = t_loop.best / t_vmap.best
print(f"loop {1e3 * t_loop.best:8.2f} ms")
print(f"vmap {1e3 * t_vmap.best:8.2f} ms   ratio {ratio:.0f}x")

💡 **What happened?** The loop pays the interpreter, and a whole reverse
sweep, once per point (Chapter 13). `vmap` runs one sweep on arrays of 10 000
entries. Nothing in `f` or `df` was written for batches.

## 3. 🧭 Jacobians: forward or reverse?

For a map $g:\mathbb R^n\to\mathbb R^m$, the full Jacobian ($m\times n$) can be
assembled in two ways (Section 3.4):

* `jacrev`: one **reverse** sweep per **output**, $m$ sweeps;
* `jacfwd`: one **forward** sweep per **input**, $n$ sweeps.

We take $g(\boldsymbol x) = \tanh(\boldsymbol A\boldsymbol x)$ with a random
matrix $\boldsymbol A$, first with **one output and 256 inputs**, then with
**256 outputs and one input**.

* `A @ x` is the matrix-vector product.
* `jacrev(g)` and `jacfwd(g)` differentiate with respect to the **first**
  argument, $\boldsymbol x$; the matrix $\boldsymbol A$ is the second argument
  and stays fixed.
* The wide matrix is divided by $\sqrt n$ so that $\boldsymbol A\boldsymbol x$
  is of order one. Otherwise $\tanh$ would be close to $\pm1$ and its
  derivative close to zero.

🤔 *Guess: which one wins in each case, and by how much?*

In [ ]:
def g(x, A):
    return torch.tanh(A @ x)


torch.manual_seed(0)
n = 256
wide = torch.randn(1, n) / n**0.5  # one output, n inputs
tall = torch.randn(n, 1)  # n outputs, one input
x_wide = torch.randn(n)
x_tall = torch.randn(1)

J_rev = jacrev(g)(x_wide, wide)
J_fwd = jacfwd(g)(x_wide, wide)
print(
    "Jacobian shape:",
    tuple(J_rev.shape),
    "| same Jacobian:",
    torch.allclose(J_rev, J_fwd),
)

t_rev_wide = %timeit -o -q jacrev(g)(x_wide, wide)
t_fwd_wide = %timeit -o -q jacfwd(g)(x_wide, wide)
t_rev_tall = %timeit -o -q jacrev(g)(x_tall, tall)
t_fwd_tall = %timeit -o -q jacfwd(g)(x_tall, tall)

print(
    f"1 output, 256 inputs:  jacrev {1e6 * t_rev_wide.best:5.0f} us   jacfwd {1e6 * t_fwd_wide.best:5.0f} us"
)
print(
    f"256 outputs, 1 input:  jacrev {1e6 * t_rev_tall.best:5.0f} us   jacfwd {1e6 * t_fwd_tall.best:5.0f} us"
)

💡 **What happened?** With one output, reverse mode needs a single sweep and
is about twice as fast; with one input, forward mode is. The gap is modest
here because each sweep is cheap and the fixed cost of a call is a large part
of the time. It grows with $n$. This is the rule of Section 3.4, and
Figure 15.1 of the notes shows it for growing $n$.

👉 A **loss** has one output and many inputs (the weights), so training always
uses reverse mode.

## 4. 🌡️ The residual, the functional way

The residual of Chapter 5 needs $u_{xx}$ for a network $u$. With `torch.func`
it is the second derivative of a function of **one** point, mapped over all
points: no seeds and no `create_graph` as in the previous notebook. The
network is built with the same seed as there, so the values should be those
of Section 7 of that notebook.

The cell below does this in three steps.

* **The network as a function of one number.** `grad` needs a function that
  takes $x$ and returns one number. A network layer `nn.Linear(1, 16)` takes a
  **batch** of points, an array of shape `(batch size, 1)`. So `u` reshapes
  the number $x$ into a batch of one point, of shape `(1, 1)`. The network
  returns an array of shape `(1, 1)`, and `.squeeze()` turns it back into one
  number.
* **The second derivative.** `grad` differentiates with respect to the
  **first** argument, here $x$. So `grad(grad(u))` is $x \mapsto u''(x)$, for
  a fixed network.
* **All points at once.** `vmap` needs to know, for each argument, which axis
  to run over. `in_dims=(0, None)` means: run over axis 0 of the first
  argument, the points `xs`, and use the same second argument, the network,
  for every point.

In [ ]:
torch.manual_seed(0)
u_net = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))


def u(x, net):
    # The network as a function of one number x.
    return net(x.reshape(1, 1)).squeeze()


u_xx = vmap(grad(grad(u)), in_dims=(0, None))
xs = torch.linspace(-1, 1, 5)
u_xx_values = u_xx(xs, u_net)
print("u_xx at five points:", u_xx_values)

💡 **What happened?** The values are those of Section 7 of the previous
notebook. The result also carries a `grad_fn`: $u_{xx}$ depends on the weights
of the network, which have `requires_grad=True`. A loss built from this
residual can therefore be differentiated with respect to the weights, as
training needs.

For a function of several variables, `hessian` gives all second derivatives,
and the Laplacian is the sum of the diagonal. Here $u(x, y) = \sin(x^2)e^y$,
written as a function of the point $p = (x, y)$, with `p[0]` $= x$ and
`p[1]` $= y$.

In [ ]:
def u2(p):
    return torch.sin(p[0] ** 2) * torch.exp(p[1])


point = torch.tensor([0.7, -0.3])
H = hessian(u2)(point)
laplacian = torch.diagonal(H).sum()  # u_xx + u_yy
print("Hessian:\n", H)
print("Laplacian:", laplacian.item())

## 🎯 Summary

| | `backward()` | `torch.func` |
|---|---|---|
| result | written into `.grad` | returned as a value |
| state | yes: results add up | none |
| derivative of a derivative | `create_graph=True` | `grad(grad(f))` |
| batch | write the loss for a batch | `vmap` |
| best for | training loops with an optimizer | derivatives inside a loss |

## 🧪 Try it yourself

1. In Section 3, use $n = 16$ instead of 256. Does the winner change? Does the
   ratio?
2. In Section 2, apply `vmap` to `d2f`. Compare with the formula of Section 1.